In [2]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib as mpl
import random
import pickle
import csv

In [4]:
def incidence_matrix(edges, nodes):

    num_edges = len(edges)
    num_nodes = len(nodes)
    
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    delta = np.zeros((num_edges, num_nodes), dtype=int)
    
    for i, (u, v) in enumerate(edges):
        delta[i, node_to_idx[u]] = -1
        delta[i, node_to_idx[v]] = 1
    
    
    return delta, node_to_idx

In [5]:
def solve_laplacian(delta, A, U, K, P, p_0, F):
    L = delta.T@A@K@delta
    N = L.shape[0]
    rhs = delta.T@A@U + F - F
     
    L_bar = L[1:, 1:]
    rhs_bar = rhs[1:] - L[1:, 0] * p_0
    P_bar = np.linalg.solve(L_bar, rhs_bar)

    P = np.zeros(N)
    P[0] = p_0
    P[1:] = P_bar

    return P, L

In [6]:
def solve_laplacian(delta, A, B, V, K, P, p_0, theta):
    
    Q_s = ((1 - theta) * B + theta * A) * V
    L = delta.T@K@delta
    N = L.shape[0]
    
    rhs = delta.T@Q_s
     
    L_bar = L[1:, 1:]
    rhs_bar = rhs[1:] - L[1:, 0] * p_0
    P_bar = np.linalg.solve(L_bar, rhs_bar)

    P = np.zeros(N)
    P[0] = p_0
    P[1:] = P_bar

    return P, L

In [7]:
def calculate_flow(delta, A, B, K, V, P, theta):
    K_1D = np.diag(K) if K.ndim == 2 else K    # force 1D
    Q_s = ((1 - theta) * B + theta * A) * V
    Q_w = Q_s - K_1D * (delta @ P)
    return Q_w

In [8]:
def calculate_core_flow(delta, A, K, V, P):
    K_1D = np.diag(K) if K.ndim == 2 else K
    Q_inner = A**2 * V - K_1D * (delta @ P)
    Q_inner = A * V - K_1D * (delta @ P)
    return Q_inner

In [9]:
def calculate_outer_flow(A, B, V, theta):
    Q_annulus = V * ((1-theta)*B + theta*A - A**2)
    Q_annulus = V * ((1-theta)*(B-A))
    return Q_annulus

In [10]:
def compute_lipid_flows(G, edges, node_to_idx, Q_total):
    """
    Computes lipid flow on each edge by the 50/50 equipartition rule.
    Q_total is the total lipid flow entering the network at the root.
    Returns Q_lipid: array of lipid flow rates per edge.
    """
    num_edges = len(edges)
    Q_lipid = np.zeros(num_edges)
    
    memo = {}
    def count_leaves(node):
        if node in memo:
            return memo[node]
        children = list(G.successors(node))
        if len(children) == 0:
            memo[node] = 1
        else:
            memo[node] = sum(count_leaves(c) for c in children)
        return memo[node]
    
    total_leaves = count_leaves(0)
    
    for e_idx, (i, j) in enumerate(edges):
        downstream = count_leaves(j)
        Q_lipid[e_idx] = Q_total * downstream / total_leaves
    
    return Q_lipid

**BUILDING COMPLEX NETWORKS**

In [11]:
def _point_segment_closest(px, py, ax, ay, bx, by):
    """Closest point on segment AB to P; returns (dist, qx, qy, t) with t in [0,1]."""
    abx, aby = bx - ax, by - ay
    apx, apy = px - ax, py - ay
    ab2 = abx * abx + aby * aby
    if ab2 < 1e-18: # tolerance for zero length edge
        return np.hypot(apx, apy), ax, ay, 0.0
    t = (apx * abx + apy * aby) / ab2
    t = max(0.0, min(1.0, t))
    qx, qy = ax + t * abx, ay + t * aby
    return np.hypot(px - qx, py - qy), qx, qy, t

In [12]:

def try_attach_tip_to_network(
    px,
    py,
    anchor,
    seg_len,
    area,
    G,
    pos,
    area_values,
    direction,
    num_nodes,
    step,
    attach_thresh,
    fusion_min_accum,
):
    """If the tip lies near an existing vertex or edge, connect, anastomose, and stop this tip.

    Returns (attached: bool, num_nodes).
    """
    if attach_thresh <= 0:
        return False, num_nodes

    best_d = attach_thresh
    mode = None  # ('node', n) or ('edge', u, v, qx, qy, t)

    for n in G.nodes():
        if n == anchor:
            continue
        nx_, ny_ = pos[n]
        d = np.hypot(px - nx_, py - ny_)
        if d < best_d:
            if G.has_edge(anchor, n) or G.has_edge(n, anchor):
                continue
            best_d = d
            mode = ("node", n)

    for u, v in G.edges():
        if anchor in (u, v) and seg_len < fusion_min_accum:
            continue
        ax_, ay_ = pos[u]
        bx_, by_ = pos[v]
        d, qx, qy, t = _point_segment_closest(px, py, ax_, ay_, bx_, by_)
        # Interior hit on an edge incident to anchor would duplicate u→join or break direction; use vertex pass.
        if anchor in (u, v) and 1e-6 < t < 1.0 - 1e-6:
            continue
        if d >= best_d:
            continue
        # if near an endpoint let the vertex test handle it
        if t <= 1e-6 or t >= 1.0 - 1e-6:
            continue
        best_d = d
        mode = ("edge", u, v, qx, qy, t)

    if mode is None:
        return False, num_nodes

    cond_a = np.random.uniform(0.4, 0.7)
    run_L = max(seg_len + best_d, step)

    if mode[0] == "node":
        n = mode[1]
        G.add_edge(anchor, n, A=area, K=cond_a, L=run_L)
        return True, num_nodes

    _, u, v, qx, qy, t = mode
    data = G[u][v]
    L_old, A_e, K_e = data["L"], data["A"], data["K"]
    ax_, ay_ = pos[u]
    bx_, by_ = pos[v]
    geom = np.hypot(bx_ - ax_, by_ - ay_)
    if geom < 1e-12:
        return False, num_nodes
    dist_u = np.hypot(qx - ax_, qy - ay_)
    dist_v = np.hypot(bx_ - qx, by_ - qy)
    L_u = max(L_old * (dist_u / geom), step)
    L_v = max(L_old * (dist_v / geom), step)

    G.remove_edge(u, v)
    join = num_nodes
    pos[join] = (qx, qy)
    direction[join] = (0.0, 0.0)
    area_values[join] = np.random.uniform(0.1, 0.3)
    G.add_node(join)
    num_nodes += 1

    G.add_edge(u, join, A=A_e, K=K_e, L=L_u)
    G.add_edge(join, v, A=A_e, K=K_e, L=L_v)
    G.add_edge(anchor, join, A=area, K=cond_a, L=run_L)
    return True, num_nodes

In [13]:
def _segments_intersect(ax, ay, bx, by, cx, cy, dx_, dy_):
    # check if two line segments intersect
    def orient(px, py, qx, qy, rx, ry):
        # returns the orientation of the triplet 
        return (qx - px) * (ry - py) - (qy - py) * (rx - px)

    def on_segment(px, py, qx, qy, rx, ry):
        # check if point rx, ry lies on the line segment px, py - qx, qy
        return (
            min(px, qx) - 1e-12 <= rx <= max(px, qx) + 1e-12
            and min(py, qy) - 1e-12 <= ry <= max(py, qy) + 1e-12
        )

    o1 = orient(ax, ay, bx, by, cx, cy)
    o2 = orient(ax, ay, bx, by, dx_, dy_)
    o3 = orient(cx, cy, dx_, dy_, ax, ay)
    o4 = orient(cx, cy, dx_, dy_, bx, by)

    if (o1 > 0) != (o2 > 0) and (o3 > 0) != (o4 > 0):
        return True

    if abs(o1) < 1e-12 and on_segment(ax, ay, bx, by, cx, cy):
        return True
    if abs(o2) < 1e-12 and on_segment(ax, ay, bx, by, dx_, dy_):
        return True
    if abs(o3) < 1e-12 and on_segment(cx, cy, dx_, dy_, ax, ay):
        return True
    if abs(o4) < 1e-12 and on_segment(cx, cy, dx_, dy_, bx, by):
        return True
    return False


def segment_conflicts(ax, ay, bx, by, anchor, G, pos, clearance):
    # check if the segment intersects with any other segment in the network
    for u, v in G.edges():
        if anchor in (u, v):
            continue
        cx, cy = pos[u]
        dx_, dy_ = pos[v]
        if _segments_intersect(ax, ay, bx, by, cx, cy, dx_, dy_):
            return True
        if _segment_segment_distance(ax, ay, bx, by, cx, cy, dx_, dy_) < clearance:
            return True
    return False

def _segment_segment_distance(ax, ay, bx, by, cx, cy, dx_, dy_):
    d1, *_ = _point_segment_closest(ax, ay, cx, cy, dx_, dy_)
    d2, *_ = _point_segment_closest(bx, by, cx, cy, dx_, dy_)
    d3, *_ = _point_segment_closest(cx, cy, ax, ay, bx, by)
    d4, *_ = _point_segment_closest(dx_, dy_, ax, ay, bx, by)
    return min(d1, d2, d3, d4)


def propose_non_overlapping_tip(
    x,
    y,
    dx,
    dy,
    anchor,
    G,
    pos,
    step,
    direction_noise,
    edge_clearance,
    proposal_retries,
    retry_turn_deg,
):
    # propose a new tip direction
    base_theta = np.arctan2(dy, dx)
    for k in range(proposal_retries):
        if k == 0:
            theta = base_theta
        else:
            theta = base_theta + np.radians(np.random.normal(0.0, retry_turn_deg))
        dx_try, dy_try = np.cos(theta), np.sin(theta)
        x_new = x + step * dx_try + np.random.normal(0, direction_noise)
        y_new = y + step * dy_try + np.random.normal(0, direction_noise)
        if segment_conflicts(x, y, x_new, y_new, anchor, G, pos, edge_clearance):
            continue
        return True, x_new, y_new, dx_try, dy_try
    return False, None, None, None, None



In [14]:

def run_simulation(seed, snapshot_time=24.0):
    """Grow a network from a single seed and solve for pressures/flows.
    
    Returns a dict with scalar summaries and full arrays.
    """
    np.random.seed(seed)
    random.seed(seed)

    # ── growth parameters ──
    dt = 1.0
    trunk_time = 8.0
    branch_prob_trunk = 0.26
    branch_prob_canopy = 0.38
    branch_decay = 0.04
    max_branches = 3
    branch_angle_t0 = 90.0
    branch_angle_tday = 63.0
    branch_angle_transition_hours = 22.0
    branch_angle_jitter = 8.0
    apical_jitter_deg = (-16, 16)
    loop_constant_factor = 0.45
    growth_rate_per_hour = 0.06
    step = growth_rate_per_hour * dt
    loop_constant = loop_constant_factor * step
    direction_noise = 0.1 * step
    edge_clearance = 0.65 * step
    proposal_retries = 10
    retry_turn_deg = 18.0
    fusion_min_accum = 2.5 * step
    attach_thresh = 0.12 * step

    def target_branch_angle_deg(t_hours):
        if t_hours >= branch_angle_transition_hours:
            return branch_angle_tday
        frac = t_hours / branch_angle_transition_hours
        return branch_angle_t0 + frac * (branch_angle_tday - branch_angle_t0)

    # ── initialise network ──
    G = nx.DiGraph()
    G.add_node(0)
    num_nodes = 1

    pos = {0: (0, 0)}
    direction = {0: (1.0, np.random.uniform(-0.14, 0.14))}
    _ = np.random.uniform(0.4, 1.0)  # preserve RNG sequence
    area_values = {0: 1.0}

    growing_tips = [(0, pos[0], direction[0], area_values[0], 0.0, 0.0)]
    time_elapsed = 0.0
    max_time = snapshot_time

    # ── growth loop ──
    while growing_tips and time_elapsed < max_time:
        next_tips = []

        for (anchor, (x, y), (dx, dy), area, tip_time, seg_len) in growing_tips:
            norm = np.sqrt(dx**2 + dy**2)
            dx /= norm
            dy /= norm

            if time_elapsed < trunk_time:
                p_fork = min(1.0, branch_prob_trunk)
            else:
                p_fork = min(
                    1.0,
                    branch_prob_canopy
                    * np.exp(-branch_decay * max(0.0, time_elapsed - trunk_time)),
                )
            if np.random.random() > p_fork:
                ok, x_new, y_new, dx_prop, dy_prop = propose_non_overlapping_tip(
                    x, y, dx, dy, anchor, G, pos,
                    step, direction_noise, edge_clearance, proposal_retries, retry_turn_deg,
                )
                if not ok:
                    continue
                dseg = np.hypot(x_new - x, y_new - y)
                seg_new = seg_len + dseg
                attached, num_nodes = try_attach_tip_to_network(
                    x_new, y_new, anchor, seg_new, area,
                    G, pos, area_values, direction, num_nodes,
                    step, attach_thresh, fusion_min_accum,
                )
                if attached:
                    continue
                next_tips.append(
                    (anchor, (x_new, y_new), (dx_prop, dy_prop), area, tip_time + dt, seg_new)
                )
                continue

            n_children = 2 if np.random.random() < 0.9 else min(3, max_branches)
            tgt = target_branch_angle_deg(time_elapsed)
            lo_lat = max(20.0, tgt - branch_angle_jitter)
            hi_lat = min(170.0, tgt + branch_angle_jitter)
            ap_lo, ap_hi = apical_jitter_deg
            apical = np.radians(np.random.uniform(ap_lo, ap_hi))
            if n_children == 2:
                lat_sign = np.random.choice([-1.0, 1.0])
                lateral = lat_sign * np.radians(np.random.uniform(lo_lat, hi_lat))
                angles = [apical, lateral]
                child_area = area * 0.9
            else:
                lat_a = np.radians(np.random.uniform(lo_lat, hi_lat))
                lat_b = -np.radians(np.random.uniform(lo_lat, hi_lat))
                angles = [apical, lat_a, lat_b]
                child_area = area * 0.9

            branch_node = num_nodes
            pos[branch_node] = (x, y)
            direction[branch_node] = (dx, dy)
            area_values[branch_node] = area
            G.add_node(branch_node)
            cond = np.random.uniform(0.4, 0.7)
            G.add_edge(anchor, branch_node, A=area, K=cond, L=max(seg_len, step))
            num_nodes += 1

            for i, angle in enumerate(angles):
                cosA, sinA = np.cos(angle), np.sin(angle)
                dx_new = dx * cosA - dy * sinA
                dy_new = dx * sinA + dy * cosA
                ok, x_new, y_new, dx_new, dy_new = propose_non_overlapping_tip(
                    x, y, dx_new, dy_new, branch_node, G, pos,
                    step, direction_noise, edge_clearance, proposal_retries, retry_turn_deg,
                )
                if not ok:
                    continue
                d0 = np.hypot(x_new - x, y_new - y)
                ca = child_area * (1.06 if i == 0 else 0.9)
                attached, num_nodes = try_attach_tip_to_network(
                    x_new, y_new, branch_node, d0, ca,
                    G, pos, area_values, direction, num_nodes,
                    step, attach_thresh, fusion_min_accum,
                )
                if attached:
                    continue
                next_tips.append(
                    (branch_node, (x_new, y_new), (dx_new, dy_new), ca, tip_time + dt, d0)
                )

        # anastomosis: merge close tips
        alive = [True] * len(next_tips)
        for i in range(len(next_tips)):
            if not alive[i]:
                continue
            anchor_i, (x1, y1), _, _, tip_time_i, seg_i = next_tips[i]
            loop_threshold = loop_constant * np.exp(-0.05 * tip_time_i)
            for j in range(i + 1, len(next_tips)):
                if not alive[j]:
                    continue
                anchor_j, (x2, y2), _, _, _, seg_j = next_tips[j]
                dist = np.hypot(x1 - x2, y1 - y2)
                if dist >= loop_threshold:
                    continue
                if (
                    anchor_i == anchor_j
                    and seg_i < fusion_min_accum
                    and seg_j < fusion_min_accum
                ):
                    continue
                if G.has_edge(anchor_i, anchor_j):
                    continue

                mx, my = (x1 + x2) / 2, (y1 + y2) / 2
                fus = num_nodes
                pos[fus] = (mx, my)
                direction[fus] = (0.0, 0.0)
                area_values[fus] = np.random.uniform(0.1, 0.3)
                G.add_node(fus)
                cond = np.random.uniform(0.4, 0.7)
                a_fus = area_values[fus]
                li = max(seg_i + np.hypot(x1 - mx, y1 - my), step)
                lj = max(seg_j + np.hypot(x2 - mx, y2 - my), step)
                G.add_edge(anchor_i, fus, A=a_fus, K=cond, L=li)
                G.add_edge(anchor_j, fus, A=a_fus, K=np.random.uniform(0.4, 0.7), L=lj)
                num_nodes += 1
                alive[i] = alive[j] = False
                break

        growing_tips = [next_tips[k] for k in range(len(next_tips)) if alive[k]]
        time_elapsed += dt

    # attach remaining free tips as terminal vertices
    for (anchor, (x, y), (dx, dy), area, _tip_time, seg_len) in growing_tips:
        attached, num_nodes = try_attach_tip_to_network(
            x, y, anchor, seg_len, area,
            G, pos, area_values, direction, num_nodes,
            step, attach_thresh, fusion_min_accum,
        )
        if attached:
            continue
        tip_node = num_nodes
        pos[tip_node] = (x, y)
        direction[tip_node] = (dx, dy)
        area_values[tip_node] = area
        G.add_node(tip_node)
        cond = np.random.uniform(0.4, 0.7)
        G.add_edge(anchor, tip_node, A=area, K=cond, L=max(seg_len, step))
        num_nodes += 1

    # ── solve for pressures & flows ──
    edges = list(G.edges)
    nodes = list(G.nodes)
    num_edges = len(edges)
    num_nodes = len(nodes)

    delta, node_to_idx = incidence_matrix(edges, nodes)

    theta = 0.1
    B = np.array([(area_values[u] + area_values[v]) / 2 for u, v in edges])
    L_edge = np.array([G[u][v]["L"] for u, v in edges])

    P = np.zeros(num_nodes)
    p0 = 0.0
    V = np.full(num_edges, 1.0)

    n_root_edges = sum(1 for u, _ in edges if u == 0)
    Q_lipid_total = theta * V[0] * B[0] * n_root_edges

    Q_lipid_raw = compute_lipid_flows(G, edges, node_to_idx, Q_lipid_total)
    A_min = 0.6 * B
    Q_lipid_cap = theta * V * (B - A_min)
    safe_den = np.maximum(Q_lipid_raw, 1e-12)
    alpha = min(1.0, np.min(Q_lipid_cap / safe_den))
    Q_lipid = alpha * Q_lipid_raw
    A_demand = B - Q_lipid / (theta * V)

    A = A_demand
    K = A**2 / L_edge

    P_sol, L_mat = solve_laplacian(delta, A, B, V, np.diag(K), P, p0, theta)
    Q_w = calculate_flow(delta, A, B, K, V, P_sol, theta)
    Q_core = calculate_core_flow(delta, A, K, V, P_sol)
    Q_annulus = calculate_outer_flow(A, B, V, theta)

    num_loops = num_edges - num_nodes + 1  # cycle rank for connected digraph

    return {
        "seed": seed,
        "snapshot_time": snapshot_time,
        "num_nodes": num_nodes,
        "num_edges": num_edges,
        "num_loops": num_loops,
        "mean_pressure": P_sol.mean(),
        "max_pressure": P_sol.max(),
        "mean_abs_Qw": np.abs(Q_w).mean(),
        "max_abs_Qw": np.abs(Q_w).max(),
        "mean_Qcore": Q_core.mean(),
        "mean_Qannulus": Q_annulus.mean(),
        "mean_speed_core": (Q_core / np.maximum(A, 1e-12)).mean(),
        "mean_speed_annulus": (Q_annulus / np.maximum(B - A, 1e-12)).mean(),
        "Q_lipid_total": Q_lipid_total,
        "alpha": alpha,
        "P_sol": P_sol.copy(),
        "Q_w": Q_w.copy(),
        "Q_core": Q_core.copy(),
        "Q_annulus": Q_annulus.copy(),
        "A": A.copy(),
        "B": B.copy(),
        "K": K.copy(),
        "edges": list(edges),
        "pos": dict(pos),
        "G": G.copy(),
    }

print("run_simulation() defined.")


run_simulation() defined.


In [ ]:
# ── configure seeds ──
N_SEEDS = 50
rng = np.random.default_rng(42)
seeds = [int(s) for s in rng.integers(0, 2**31, size=N_SEEDS)]

SNAPSHOT_TIME = 24.0

# ── run all seeds ──
all_results = []
failed_seeds = []

for idx, s in enumerate(seeds):
    try:
        res = run_simulation(s, snapshot_time=SNAPSHOT_TIME)
        all_results.append(res)
    except Exception as e:
        failed_seeds.append((s, str(e)))
    if (idx + 1) % 10 == 0 or idx == len(seeds) - 1:
        print(f"  completed {idx + 1}/{len(seeds)}  (failures so far: {len(failed_seeds)})")

print(f"\nDone. {len(all_results)} succeeded, {len(failed_seeds)} failed.")

# build CSV summary (no pandas needed)
fields = [
    "seed", "snapshot_time", "num_nodes", "num_edges", "num_loops",
    "mean_pressure", "max_pressure", "mean_abs_Qw", "max_abs_Qw",
    "mean_Qcore", "mean_Qannulus", "mean_speed_core", "mean_speed_annulus",
    "Q_lipid_total", "alpha",
]

with open("seed_summary.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    for r in all_results:
        writer.writerow({k: r[k] for k in fields})

print(f"Saved seed_summary.csv  ({len(all_results)} rows)")

# quick preview of first few rows
print(f"\n{'seed':>12s} {'nodes':>5s} {'edges':>5s} {'loops':>5s} {'mean_P':>10s} {'max|Qw|':>10s}")
for r in all_results[:10]:
    print(f"{r['seed']:12d} {r['num_nodes']:5d} {r['num_edges']:5d} {r['num_loops']:5d} {r['mean_pressure']:10.4f} {r['max_abs_Qw']:10.6f}")

print(failed_seeds[:3])

  completed 10/50  (failures so far: 0)
  completed 20/50  (failures so far: 0)
  completed 30/50  (failures so far: 0)
  completed 40/50  (failures so far: 0)
  completed 50/50  (failures so far: 0)

Done. 50 succeeded, 0 failed.
Saved seed_summary.csv  (50 rows)
Saved seed_results_full.pkl

        seed nodes edges loops     mean_P    max|Qw|
   191664964   261   268     8     2.4543   0.123595
  1662057958   111   115     5     2.6252   0.181053
  1405681632   236   246    11     2.3176   0.193187
   942484272   232   241    10     2.3626   0.129392
   929893138   263   268     6     2.3678   0.107650
  1843824993   246   250     5     2.4337   0.100086
   184566854   233   236     4     2.3948   0.131077
  1497586439   295   303     9     2.4186   0.122936
   432652533   344   351     8     2.9703   0.194126
   202244314   152   155     4     2.2265   0.056769
[]


In [15]:
snapshot_times = [0.0, 5.0, 10.0, 15.0, 20.0, 25.0, 30.0, 35.0, 40.0, 45.0, 50.0, 55.0, 60.0, 65.0, 70.0, 75.0, 80.0]
snapshot_times = [0.0, 20.0, 40.0, 60.0, 80.0, 100.0, 120.0]
N_SEEDS = 1
rng = np.random.default_rng(42)
seeds = [int(s) for s in rng.integers(0, 2**31, size=N_SEEDS)]

all_results = []
failed = []

for t in snapshot_times:
    for idx, s in enumerate(seeds):
        try:
            res = run_simulation(s, snapshot_time=t)
            all_results.append(res)
        except Exception as e:
            failed.append((s, t, str(e)))
    print(f"  t={t:.1f}h done  ({len(all_results)} total, {len(failed)} failed)")

# build CSV summary
fields = [
    "seed", "snapshot_time", "num_nodes", "num_edges", "num_loops",
    "mean_pressure", "max_pressure", "mean_abs_Qw", "max_abs_Qw",
    "mean_Qcore", "mean_Qannulus", "mean_speed_core", "mean_speed_annulus",
    "Q_lipid_total", "alpha",
]

with open("seed_summary_4.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    for r in all_results:
        writer.writerow({k: r[k] for k in fields})

print(f"Saved seed summary  ({len(all_results)} rows)")


  t=0.0h done  (1 total, 0 failed)
  t=20.0h done  (2 total, 0 failed)
  t=40.0h done  (3 total, 0 failed)
  t=60.0h done  (4 total, 0 failed)
  t=80.0h done  (5 total, 0 failed)
  t=100.0h done  (6 total, 0 failed)
  t=120.0h done  (7 total, 0 failed)
Saved seed summary  (7 rows)
